# nexus-decision-4b: LoRA distillation on Google Colab

This notebook runs **only the training step** of the pipeline, on an A100 (recommended) or an L4.
Data generation, evaluation, calibration and release run locally.

**Before you start**
1. On your machine, run `uv run python -m data.build_splits`, then `scripts/pack_colab.sh`.
2. Upload `artifacts/colab/nexus-colab.zip` to Google Drive as `MyDrive/nexus-decision-4b/nexus-colab.zip`.
3. In Colab, choose *Runtime → Change runtime type → A100 GPU* (or L4).

Checkpoints are written to Google Drive every 200 optimizer steps. If the runtime disconnects,
run every cell again; the training cell resumes from the last checkpoint.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/nexus-decision-4b'
RUN = 'run1'                      # change to start a fresh run in a new folder
OUT = f'{DRIVE}/adapters/{RUN}'
WORK = '/content/nexus'

!mkdir -p {WORK} {OUT}
!cd {WORK} && unzip -o -q {DRIVE}/nexus-colab.zip && ls

## Install

The pins match the versions the pipeline was tested with. `flash-linear-attention` provides the
fast Gated DeltaNet kernels. Without it, training is many times slower.

In [ ]:
!pip install -q "transformers==5.22.2" "peft==0.21.2" "accelerate>=1.0" pyyaml flash-linear-attention
# Optional: a faster causal conv1d kernel. Building it can take several minutes and it is not required.
# !pip install -q causal-conv1d --no-build-isolation

In [ ]:
import importlib.util, torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| bf16', torch.cuda.is_bf16_supported())
print('flash-linear-attention installed:', importlib.util.find_spec('fla') is not None)
gpu = torch.cuda.get_device_name(0)
CONFIG = 'train/configs/colab_a100.yaml' if 'A100' in gpu or 'H100' in gpu else 'train/configs/colab_l4.yaml'
print(gpu, '->', CONFIG)

## 1. Throughput check (about 2 minutes)

Runs 10 optimizer steps into a scratch folder and prints `tok_per_s` and `eta_h`. The ETA covers
the whole run. Check that no line says *falling back to its reference PyTorch implementation* for
`chunk_gated_delta_rule`.

In [ ]:
!cd {WORK} && python train/train_torch.py --config {CONFIG} --max-steps 10 \
    --set out_dir=/content/bench report_every=5 eval_every=100000 save_every=100000 2>&1 | grep -v Warning

## 2. Full training

Resumable: rerun this cell after a disconnect. Keep the browser tab open (Colab Pro stops idle
sessions). Progress is also appended to `train_log.jsonl` in the Drive folder.

In [ ]:
!cd {WORK} && python train/train_torch.py --config {CONFIG} --resume --set out_dir={OUT} 2>&1 \
    | grep -v Warning | tee -a {OUT}/console.log

## 3. Result

The trained adapter is in `MyDrive/nexus-decision-4b/adapters/run1/final`
(`adapter_model.safetensors` plus `adapter_config.json`, about 100 MB).
Download that folder to the project as `artifacts/adapters/colab/final`, then on your machine run:

```bash
uv run python train/fuse_export.py --adapters artifacts/adapters/colab/final \
    --out artifacts/fused/nexus-decision-4b
uv run python -m eval.run_eval --name tuned --model artifacts/fused/nexus-decision-4b --split calib
uv run python -m eval.run_eval --name tuned --model artifacts/fused/nexus-decision-4b --split test
uv run python -m eval.report --name tuned
```

In [ ]:
import json
log = [json.loads(l) for l in open(f'{OUT}/train_log.jsonl')]
for r in log[-8:]:
    print(r)
!ls -la {OUT}/final